In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from jax import vmap
from src.fdm import ECirreMechanismFDMSolver
from src.params import ECirreMechanismFDMParams
from src.voltammetry import LinearSweepDC

plt.style.use("seaborn-v0_8-darkgrid")

# Effect of parameters

In [ ]:
voltammetry = LinearSweepDC()
fdm_solver = ECirreMechanismFDMSolver(voltammetry)
base_params = ECirreMechanismFDMParams(
    alpha=jnp.array(0.6),
    K0=jnp.array(1.0),
    Kplus=jnp.array(1.0),
    Kminus=jnp.array(1.0),
    E0=jnp.array(2.0),
    dB=jnp.array(0.5),
)

kplus_range = jnp.array([1.0, 2.0, 5.0, 10.0, 20.0, 50.0])

kplus_params = ECirreMechanismFDMParams(
    alpha=jnp.full_like(kplus_range, base_params.alpha),
    K0=jnp.full_like(kplus_range, base_params.K0),
    Kplus=kplus_range,
    Kminus=jnp.full_like(kplus_range, base_params.Kminus),
    E0=jnp.full_like(kplus_range, base_params.E0),
    dB=jnp.full_like(kplus_range, base_params.dB),
)

kplus_currents = vmap(fdm_solver.solve)(kplus_params)

kminus_range = jnp.array([1.0, 2.0, 5.0, 10.0, 20.0, 50.0])

kminus_params = ECirreMechanismFDMParams(
    alpha=jnp.full_like(kminus_range, base_params.alpha),
    K0=jnp.full_like(kminus_range, base_params.K0),
    Kplus=jnp.full_like(kminus_range, base_params.Kplus),
    Kminus=kminus_range,
    E0=jnp.full_like(kminus_range, base_params.E0),
    dB=jnp.full_like(kminus_range, base_params.dB),
)

kminus_currents = vmap(fdm_solver.solve)(kminus_params)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

for kp, c in zip(kplus_range, kplus_currents):
    ax1.plot(fdm_solver.applied_potentials, c, label=kp)

ax1.xaxis.set_inverted(True)
ax1.yaxis.set_inverted(True)
ax1.set_title(r"$K_1$")
ax1.legend()

for km, c in zip(kminus_range, kminus_currents):
    ax2.plot(fdm_solver.applied_potentials, c, label=km)

ax2.xaxis.set_inverted(True)
ax2.yaxis.set_inverted(True)
ax2.set_title(r"$K_{-1}$")
ax2.legend()

plt.show()

# Analytical Results

In [ ]:
params = ECirreMechanismFDMParams(
    alpha=jnp.array(1.0),
    K0=jnp.array(1000.0),
    Kminus=jnp.array(1.0),
    Kplus=jnp.array(1000.0),
    E0=jnp.array(0.0),
    dB=jnp.array(1.0),
)
h = 1e-3
dtheta = 1e-3

voltammetry = LinearSweepDC()

fdm_solver = ECirreMechanismFDMSolver(voltammetry, h, dtheta)

current = fdm_solver.solve(params)

ss_current = -jnp.sqrt(params.Kplus) / (1 + jnp.exp(fdm_solver.applied_potentials))

plt.plot(fdm_solver.applied_potentials, ss_current, label="Analytical SS")
plt.plot(fdm_solver.applied_potentials, current, label="FDM")
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.legend()
plt.show()


In [ ]:
import numpy as np

rw_mh = np.load("./data/ECirre_MetropolisHastings_LinearSweepDC.npz")
hmc = np.load("./data/ECirre_HMC_LinearSweepDC.npz")

hist_kwargs = dict(
    alpha=0.75,
    bins=50,
    density=True,
)

true_params = ECirreMechanismFDMParams(
    alpha=jnp.array(0.6),
    K0=jnp.array(20.0),
    Kplus=jnp.array(10.0),
    Kminus=jnp.array(1.0),
    E0=jnp.array(2.0),
    dB=jnp.array(1.2),
)


fig, axs = plt.subplots(2, 3, figsize=(12, 6))

axs[0, 0].hist(rw_mh["alpha"].flatten(), **hist_kwargs, label="RW MH")
axs[0, 0].hist(hmc["alpha"].flatten(), **hist_kwargs, label="HMC")
axs[0, 0].axvline(x=true_params.alpha, linestyle="--", c="black", label="True Value")

axs[0, 1].hist(rw_mh["K0"].flatten(), **hist_kwargs)
axs[0, 1].hist(hmc["K0"].flatten(), **hist_kwargs)
axs[0, 1].axvline(x=true_params.K0, linestyle="--", c="black")

axs[0, 2].hist(rw_mh["Kplus"].flatten(), **hist_kwargs)
axs[0, 2].hist(hmc["Kplus"].flatten(), **hist_kwargs)
axs[0, 2].axvline(x=true_params.Kplus, linestyle="--", c="black")


axs[1, 0].hist(rw_mh["Kminus"].flatten(), **hist_kwargs)
axs[1, 0].hist(hmc["Kminus"].flatten(), **hist_kwargs)
axs[1, 0].axvline(x=true_params.Kminus, linestyle="--", c="black")

axs[1, 1].hist(rw_mh["E0"].flatten(), **hist_kwargs)
axs[1, 1].hist(hmc["E0"].flatten(), **hist_kwargs)
axs[1, 1].axvline(x=true_params.E0, linestyle="--", c="black")

axs[1, 2].hist(rw_mh["dB"].flatten(), **hist_kwargs)
axs[1, 2].hist(hmc["dB"].flatten(), **hist_kwargs)
axs[1, 2].axvline(x=true_params.dB, linestyle="--", c="black")

handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=3,
    frameon=False,
)

plt.show()